# 11 - Pest Symptom Classifier (YOLOv8 Classification)

This notebook trains a YOLOv8 classification model for pest
symptom classification.

The trained model and all model-related training artifacts are
stored inside:

    models/pest_classifier/

Source datasets are stored inside:

    Final Datasets/

Generated train/validation/test data is stored inside:

    pest_dataset/

## Import and configurations

In [1]:
# ============================================================
# IMPORTS & CONFIGURATION
# ============================================================

from pathlib import Path
import shutil
import random

import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import train_test_split

from ultralytics import YOLO


# ============================================================
# PROJECT PATHS
# ============================================================

# Current notebook directory
#
# Expected:
# Z:\Projects\Smart-Farming\Crop Identification\notebooks

NOTEBOOK_DIR = Path.cwd()


# Project root
#
# Z:\Projects\Smart-Farming\Crop Identification

PROJECT_ROOT = NOTEBOOK_DIR.parent


# Original datasets
#
# Z:\Projects\Smart-Farming\Crop Identification\Final Datasets

FINAL_DATASETS_DIR = (
    PROJECT_ROOT /
    "Final Datasets"
)


# Manifest file
#
# Z:\Projects\Smart-Farming\Crop Identification\notebooks\manifest.csv

MANIFEST_PATH = (
    NOTEBOOK_DIR /
    "manifest.csv"
)


# Generated dataset
#
# Z:\Projects\Smart-Farming\Crop Identification\notebooks\pest_dataset

OUT_DIR = (
    NOTEBOOK_DIR /
    "pest_dataset"
)


# ============================================================
# MODEL DIRECTORY
# ============================================================

# EVERYTHING related to the trained pest classifier
# will be stored inside this directory.
#
# Z:\Projects\Smart-Farming\Crop Identification\
#     notebooks\models\pest_classifier

MODELS_DIR = (
    NOTEBOOK_DIR /
    "models" /
    "pest_classifier"
)


# Final deployment-ready model
#
# models/
# └── pest_classifier/
#     └── pest_classifier.pt

FINAL_MODEL_PATH = (
    MODELS_DIR /
    "pest_classifier.pt"
)


# Final last checkpoint
FINAL_LAST_MODEL_PATH = (
    MODELS_DIR /
    "last.pt"
)


# Ultralytics training artifacts
#
# models/
# └── pest_classifier/
#     └── training/
#         └── run/

TRAIN_OUTPUT_DIR = (
    MODELS_DIR /
    "training"
)


# ============================================================
# TRAINING SETTINGS
# ============================================================

MIN_CLASS_COUNT = 15

RANDOM_SEED = 42

TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

EPOCHS = 30
IMAGE_SIZE = 224


# ============================================================
# PEST CLASS MAPPING
# ============================================================

PEST_CANONICAL_MAP = {
    "A": "A",
    "AW": "AW",
    "LM": "LM",
    "SM": "SM",
    "T": "T",
}


# ============================================================
# CREATE DIRECTORIES
# ============================================================

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRAIN_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# PRINT CONFIGURATION
# ============================================================

print("=" * 70)
print("PROJECT CONFIGURATION")
print("=" * 70)

print(f"Notebook directory : {NOTEBOOK_DIR}")
print(f"Project root       : {PROJECT_ROOT}")
print(f"Final Datasets     : {FINAL_DATASETS_DIR}")
print(f"Manifest           : {MANIFEST_PATH}")
print(f"Dataset output     : {OUT_DIR}")
print(f"Models directory   : {MODELS_DIR}")
print(f"Training output    : {TRAIN_OUTPUT_DIR}")
print(f"Final model        : {FINAL_MODEL_PATH}")

print("=" * 70)

PROJECT CONFIGURATION
Notebook directory : z:\Projects\Smart-Farming\Crop Identification\notebooks
Project root       : z:\Projects\Smart-Farming\Crop Identification
Final Datasets     : z:\Projects\Smart-Farming\Crop Identification\Final Datasets
Manifest           : z:\Projects\Smart-Farming\Crop Identification\notebooks\manifest.csv
Dataset output     : z:\Projects\Smart-Farming\Crop Identification\notebooks\pest_dataset
Models directory   : z:\Projects\Smart-Farming\Crop Identification\notebooks\models\pest_classifier
Training output    : z:\Projects\Smart-Farming\Crop Identification\notebooks\models\pest_classifier\training
Final model        : z:\Projects\Smart-Farming\Crop Identification\notebooks\models\pest_classifier\pest_classifier.pt


## Verify Project structure

In [2]:
# ============================================================
# VERIFY PROJECT STRUCTURE
# ============================================================

print("=" * 70)
print("VERIFYING PROJECT STRUCTURE")
print("=" * 70)


# ------------------------------------------------------------
# Check Final Datasets
# ------------------------------------------------------------

if not FINAL_DATASETS_DIR.exists():

    raise FileNotFoundError(
        "\nFinal Datasets folder was not found.\n\n"
        f"Expected:\n{FINAL_DATASETS_DIR}\n\n"
        "Please check your project structure."
    )


# ------------------------------------------------------------
# Check manifest
# ------------------------------------------------------------

if not MANIFEST_PATH.exists():

    raise FileNotFoundError(
        "\nmanifest.csv was not found.\n\n"
        f"Expected:\n{MANIFEST_PATH}"
    )


# ------------------------------------------------------------
# Print paths
# ------------------------------------------------------------

print("\nFinal Datasets:")
print(FINAL_DATASETS_DIR)

print("\nManifest:")
print(MANIFEST_PATH)

print("\nModels directory:")
print(MODELS_DIR)


# ------------------------------------------------------------
# Show dataset folders
# ------------------------------------------------------------

print("\nAvailable dataset folders:")

for folder in sorted(
    FINAL_DATASETS_DIR.iterdir()
):

    if folder.is_dir():

        print(
            f"  └── {folder.name}"
        )


print("\nProject structure verified successfully.")

VERIFYING PROJECT STRUCTURE

Final Datasets:
z:\Projects\Smart-Farming\Crop Identification\Final Datasets

Manifest:
z:\Projects\Smart-Farming\Crop Identification\notebooks\manifest.csv

Models directory:
z:\Projects\Smart-Farming\Crop Identification\notebooks\models\pest_classifier

Available dataset folders:
  └── CLCE
  └── CLUE
  └── GLCDfGLUE
  └── GLFCE
  └── GLUE
  └── PBCE
  └── PBUE
  └── PLCE
  └── PLUE
  └── TLCE
  └── TLUE

Project structure verified successfully.


## Build manifest and fix old paths

In [3]:
# ============================================================
# BUILD PEST MANIFEST
# ============================================================

def build_pest_manifest():

    print("\n" + "=" * 70)
    print("BUILDING PEST MANIFEST")
    print("=" * 70)


    # --------------------------------------------------------
    # Read manifest
    # --------------------------------------------------------

    df = pd.read_csv(
        MANIFEST_PATH
    )

    print(
        f"\nTotal rows in manifest: {len(df)}"
    )


    # --------------------------------------------------------
    # Check required columns
    # --------------------------------------------------------

    required_columns = {
        "filepath",
        "disease_raw"
    }

    missing_columns = (
        required_columns -
        set(df.columns)
    )

    if missing_columns:

        raise ValueError(
            "\nmanifest.csv is missing required columns:\n"
            f"{missing_columns}\n\n"
            f"Available columns:\n"
            f"{list(df.columns)}"
        )


    # --------------------------------------------------------
    # Keep only pest classes
    # --------------------------------------------------------

    df = df[
        df["disease_raw"].isin(
            PEST_CANONICAL_MAP.keys()
        )
    ].copy()


    print(
        f"Pest-related rows: {len(df)}"
    )


    # --------------------------------------------------------
    # Create canonical pest label
    # --------------------------------------------------------

    df["pest"] = (
        df["disease_raw"]
        .map(PEST_CANONICAL_MAP)
    )


    # --------------------------------------------------------
    # Fix old absolute paths
    # --------------------------------------------------------

    def fix_filepath(filepath):

        old_path = Path(
            str(filepath)
        )


        # ----------------------------------------------------
        # If original path already works,
        # don't change it.
        # ----------------------------------------------------

        if old_path.exists():

            return old_path


        # ----------------------------------------------------
        # Split path into components
        # ----------------------------------------------------

        parts = old_path.parts


        # ----------------------------------------------------
        # Find "Final Datasets"
        # ----------------------------------------------------

        try:

            index = parts.index(
                "Final Datasets"
            )

        except ValueError:

            return old_path


        # ----------------------------------------------------
        # Keep everything AFTER "Final Datasets"
        #
        # Example:
        #
        # CLUE\A\CLUE_A_01.jpg
        # ----------------------------------------------------

        relative_path = Path(
            *parts[index + 1:]
        )


        # ----------------------------------------------------
        # Rebuild using current project path
        #
        # Old:
        # Z:\Crop Identification\Final Datasets\...
        #
        # New:
        # Z:\Projects\Smart-Farming\
        # Crop Identification\Final Datasets\...
        # ----------------------------------------------------

        new_path = (
            FINAL_DATASETS_DIR /
            relative_path
        )


        return new_path


    # Apply path correction

    df["filepath"] = (
        df["filepath"]
        .apply(fix_filepath)
    )


    # --------------------------------------------------------
    # Validate all files
    # --------------------------------------------------------

    print("\nChecking image paths...")

    exists_mask = (
        df["filepath"]
        .apply(
            lambda x: Path(x).exists()
        )
    )

    missing_df = df[
        ~exists_mask
    ]


    if len(missing_df) > 0:

        print(
            f"\nERROR: {len(missing_df)} "
            "images were not found."
        )

        print(
            "\nFirst 20 missing files:"
        )

        for path in (
            missing_df["filepath"]
            .head(20)
        ):

            print(
                f"  {path}"
            )


        raise FileNotFoundError(
            f"\n{len(missing_df)} images "
            "could not be found.\n\n"
            f"Dataset root:\n"
            f"{FINAL_DATASETS_DIR}"
        )


    print(
        f"All {len(df)} image paths are valid."
    )


    # --------------------------------------------------------
    # Class distribution
    # --------------------------------------------------------

    print(
        "\nCombined pest class counts "
        "(across all crops):"
    )

    counts = (
        df["pest"]
        .value_counts()
        .sort_values(
            ascending=False
        )
    )

    print(counts)


    return df

## Remove rare classes

In [4]:
# ============================================================
# REMOVE RARE CLASSES
# ============================================================

def prune_rare_classes(
    df,
    min_count=MIN_CLASS_COUNT
):

    counts = (
        df["pest"]
        .value_counts()
    )


    rare_classes = (
        counts[
            counts < min_count
        ]
        .index
        .tolist()
    )


    if rare_classes:

        print(
            f"\nDropping rare classes "
            f"(< {min_count} images): "
            f"{rare_classes}"
        )

        print(
            "These classes don't have enough "
            "images to train reliably."
        )


        df = df[
            ~df["pest"].isin(
                rare_classes
            )
        ].copy()


    else:

        print(
            f"\nNo classes have fewer than "
            f"{min_count} images."
        )


    print(
        "\nFinal class distribution:"
    )

    print(
        df["pest"]
        .value_counts()
        .sort_values(
            ascending=False
        )
    )


    return df

## Create train/val/test dataset

In [5]:
# ============================================================
# SPLIT DATASET AND MATERIALIZE FILES
# ============================================================

def split_and_materialize(
    df,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    test_size=TEST_SIZE,
    seed=RANDOM_SEED
):

    print("\n" + "=" * 70)
    print("PREPARING TRAIN / VALIDATION / TEST DATASET")
    print("=" * 70)


    # --------------------------------------------------------
    # Validate source images
    # --------------------------------------------------------

    print("\nChecking source images...")

    missing_files = []


    for path in tqdm(
        df["filepath"],
        desc="Checking files"
    ):

        if not Path(path).exists():

            missing_files.append(
                str(path)
            )


    if missing_files:

        print(
            f"\nERROR: {len(missing_files)} "
            "files are missing."
        )

        print(
            "\nFirst 20 missing files:"
        )

        for path in (
            missing_files[:20]
        ):

            print(
                f"  {path}"
            )


        raise FileNotFoundError(
            "Some source images do not exist."
        )


    print(
        f"All {len(df)} source images exist."
    )


    # --------------------------------------------------------
    # Remove previous generated dataset
    # --------------------------------------------------------

    if OUT_DIR.exists():

        print(
            f"\nRemoving old dataset:\n"
            f"{OUT_DIR}"
        )

        shutil.rmtree(
            OUT_DIR
        )


    OUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )


    # --------------------------------------------------------
    # Train / temporary split
    # --------------------------------------------------------

    train_df, temp_df = (
        train_test_split(
            df,
            train_size=train_size,
            stratify=df["pest"],
            random_state=seed
        )
    )


    # --------------------------------------------------------
    # Validation / test split
    # --------------------------------------------------------

    relative_val_size = (
        val_size /
        (val_size + test_size)
    )


    val_df, test_df = (
        train_test_split(
            temp_df,
            train_size=relative_val_size,
            stratify=temp_df["pest"],
            random_state=seed
        )
    )


    # --------------------------------------------------------
    # Print split sizes
    # --------------------------------------------------------

    print("\nDataset split:")

    print(
        f"Train: {len(train_df)} "
        f"({len(train_df) / len(df) * 100:.2f}%)"
    )

    print(
        f"Val:   {len(val_df)} "
        f"({len(val_df) / len(df) * 100:.2f}%)"
    )

    print(
        f"Test:  {len(test_df)} "
        f"({len(test_df) / len(df) * 100:.2f}%)"
    )


    # --------------------------------------------------------
    # Copy images
    # --------------------------------------------------------

    splits = {
        "train": train_df,
        "val": val_df,
        "test": test_df
    }


    for split_name, split_df in (
        splits.items()
    ):

        print(
            f"\nCreating {split_name} dataset..."
        )


        for row_number, (_, row) in enumerate(
            tqdm(
                split_df.iterrows(),
                total=len(split_df),
                desc=f"Copying {split_name}"
            )
        ):

            src = Path(
                str(row["filepath"])
            )


            pest = str(
                row["pest"]
            )


            dest_dir = (
                OUT_DIR /
                split_name /
                pest
            )


            dest_dir.mkdir(
                parents=True,
                exist_ok=True
            )


            # ------------------------------------------------
            # Avoid duplicate filenames
            # ------------------------------------------------

            destination = (
                dest_dir /
                src.name
            )


            if destination.exists():

                stem = src.stem
                suffix = src.suffix

                counter = 1


                while True:

                    destination = (
                        dest_dir /
                        f"{stem}_{counter}{suffix}"
                    )


                    if not destination.exists():

                        break


                    counter += 1


            shutil.copy2(
                src,
                destination
            )


    # --------------------------------------------------------
    # Print generated structure
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("DATASET PREPARATION COMPLETE")
    print("=" * 70)


    print(
        f"\nDataset location:\n"
        f"{OUT_DIR.resolve()}"
    )


    print(
        "\nGenerated dataset:"
    )


    for split in (
        "train",
        "val",
        "test"
    ):

        split_dir = (
            OUT_DIR /
            split
        )


        print(
            f"\n{split}/"
        )


        if split_dir.exists():

            for class_dir in sorted(
                split_dir.iterdir()
            ):

                if class_dir.is_dir():

                    count = len(
                        list(
                            class_dir.iterdir()
                        )
                    )


                    print(
                        f"    {class_dir.name}: "
                        f"{count} images"
                    )


    return (
        train_df,
        val_df,
        test_df
    )

## Train model and save ALL model files inside `models`

In [6]:
# ============================================================
# TRAIN YOLOv8 CLASSIFIER
# CUDA-SAFE VERSION FOR WINDOWS
# ============================================================

def train_pest_classifier(
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE
):

    import gc
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("\n" + "=" * 70)
    print("STARTING YOLOv8 PEST CLASSIFIER TRAINING")
    print("=" * 70)

    # --------------------------------------------------------
    # MODEL OUTPUT DIRECTORIES
    # --------------------------------------------------------

    # Remove previous training artifacts
    if TRAIN_OUTPUT_DIR.exists():

        print(
            "\nRemoving previous training artifacts..."
        )

        shutil.rmtree(
            TRAIN_OUTPUT_DIR
        )

    TRAIN_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Load pretrained YOLO classification model
    # --------------------------------------------------------

    print(
        "\nLoading YOLOv8 classification model..."
    )

    model = YOLO(
        "yolov8n-cls.pt"
    )

    # --------------------------------------------------------
    # Training output directory
    # --------------------------------------------------------

    training_run_dir = (
        TRAIN_OUTPUT_DIR /
        "run"
    )

    print(
        "\nTraining configuration:"
    )

    print(
        f"  Dataset       : {OUT_DIR.resolve()}"
    )

    print(
        f"  Epochs        : {epochs}"
    )

    print(
        f"  Image size    : {imgsz}"
    )

    print(
        f"  Batch size    : 16"
    )

    print(
        f"  Device        : CUDA GPU 0"
    )

    print(
        f"  Workers       : 0"
    )

    print(
        f"  AMP           : False"
    )

    print(
        f"  Output        : {training_run_dir.resolve()}"
    )

    # --------------------------------------------------------
    # TRAIN
    #
    # IMPORTANT:
    #
    # workers=0
    #     Prevents Windows multiprocessing DataLoader issues.
    #
    # pin_memory (unsupported in this YOLO version)
    #     Fixes the CUDA "resource already mapped" error.
    #
    # cache=False
    #     Prevents unnecessary RAM/VRAM caching.
    #
    # amp=False
    #     Initially disables mixed precision for stability.
    # --------------------------------------------------------

    print(
        "\nStarting training..."
    )

    results = model.train(

        # Dataset
        data=str(
            OUT_DIR.resolve()
        ),

        # Training
        epochs=epochs,

        imgsz=imgsz,

        batch=16,

        # GPU
        device=0,

        # Windows/CUDA stability
        workers=0,

        cache=False,

        amp=False,

        # Output
        project=str(
            TRAIN_OUTPUT_DIR.resolve()
        ),

        name="run",

        exist_ok=True,

        # Reproducibility
        seed=RANDOM_SEED
    )

    # --------------------------------------------------------
    # LOCATE TRAINED WEIGHTS
    # --------------------------------------------------------

    best_weights = (
        training_run_dir /
        "weights" /
        "best.pt"
    )

    last_weights = (
        training_run_dir /
        "weights" /
        "last.pt"
    )

    # --------------------------------------------------------
    # VERIFY BEST MODEL
    # --------------------------------------------------------

    if not best_weights.exists():

        raise FileNotFoundError(
            "\nTraining completed, but best.pt "
            "was not found.\n\n"
            f"Expected location:\n"
            f"{best_weights}"
        )

    # --------------------------------------------------------
    # COPY BEST MODEL TO FINAL LOCATION
    # --------------------------------------------------------

    shutil.copy2(
        best_weights,
        FINAL_MODEL_PATH
    )

    # --------------------------------------------------------
    # COPY LAST MODEL
    # --------------------------------------------------------

    if last_weights.exists():

        shutil.copy2(
            last_weights,
            FINAL_LAST_MODEL_PATH
        )

    # --------------------------------------------------------
    # FINAL OUTPUT
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("MODEL TRAINING COMPLETE")
    print("=" * 70)

    print(
        f"\nTraining artifacts:"
        f"\n{training_run_dir.resolve()}"
    )

    print(
        f"\nBest YOLO weights:"
        f"\n{best_weights.resolve()}"
    )

    print(
        f"\nFinal deployment model:"
        f"\n{FINAL_MODEL_PATH.resolve()}"
    )

    if FINAL_LAST_MODEL_PATH.exists():

        print(
            f"\nLast checkpoint:"
            f"\n{FINAL_LAST_MODEL_PATH.resolve()}"
        )

    print(
        "\nAll model-related files are inside:"
    )

    print(
        f"{MODELS_DIR.resolve()}"
    )

    return (
        model,
        results
    )

## Evaluate model

In [7]:
# ============================================================
# EVALUATE MODEL ON TEST DATA
# CUDA-SAFE VERSION
# ============================================================

def evaluate_on_test(model):

    print("\n" + "=" * 70)
    print("EVALUATING MODEL ON TEST DATA")
    print("=" * 70)

    test_dir = (
        OUT_DIR /
        "test"
    )

    if not test_dir.exists():

        raise FileNotFoundError(
            f"\nTest directory not found:\n"
            f"{test_dir}"
        )

    print(
        f"\nTest dataset:"
        f"\n{test_dir.resolve()}"
    )

    # --------------------------------------------------------
    # Evaluation
    #
    # Same CUDA-safe settings used during training.
    # --------------------------------------------------------

    results = model.val(

        data=str(
            OUT_DIR.resolve()
        ),

        split="test",

        device=0,

        workers=0,

        batch=16,

        cache=False,

        # Important for the CUDA pinned-memory issue
        # encountered during your training.
        # Ultralytics may not expose pin_memory directly
        # during validation in every version, so workers=0
        # and cache=False are the important safeguards here.
    )

    print("\n" + "=" * 70)
    print("TEST EVALUATION COMPLETE")
    print("=" * 70)

    return results

## Run complete pipeline

In [8]:
# ============================================================
# COMPLETE TRAINING PIPELINE
# ============================================================

print("\n" + "=" * 70)
print("PEST CLASSIFIER PIPELINE")
print("=" * 70)


# ------------------------------------------------------------
# STEP 1
# Build manifest and fix paths
# ------------------------------------------------------------

df = build_pest_manifest()


# ------------------------------------------------------------
# STEP 2
# Remove rare classes
# ------------------------------------------------------------

df = prune_rare_classes(
    df
)


# ------------------------------------------------------------
# STEP 3
# Create train / validation / test dataset
# ------------------------------------------------------------

train_df, val_df, test_df = (
    split_and_materialize(
        df
    )
)


# ------------------------------------------------------------
# STEP 4
# Train model
# ------------------------------------------------------------

model, train_results = (
    train_pest_classifier(
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE
    )
)


# ------------------------------------------------------------
# STEP 5
# Evaluate
# ------------------------------------------------------------

test_results = (
    evaluate_on_test(
        model
    )
)


# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n\n")

print("=" * 70)
print("PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 70)


print(
    f"\nTraining images   : "
    f"{len(train_df)}"
)


print(
    f"Validation images : "
    f"{len(val_df)}"
)


print(
    f"Test images       : "
    f"{len(test_df)}"
)


print(
    f"\nModel directory:\n"
    f"{MODELS_DIR.resolve()}"
)


print(
    f"\nFinal model:\n"
    f"{FINAL_MODEL_PATH.resolve()}"
)


print(
    f"\nFinal model exists: "
    f"{FINAL_MODEL_PATH.exists()}"
)


print(
    f"\nLast checkpoint exists: "
    f"{FINAL_LAST_MODEL_PATH.exists()}"
)


print("=" * 70)


PEST CLASSIFIER PIPELINE

BUILDING PEST MANIFEST

Total rows in manifest: 30404
Pest-related rows: 546

Checking image paths...
All 546 image paths are valid.

Combined pest class counts (across all crops):
pest
LM    389
A      64
SM     50
AW     40
T       3
Name: count, dtype: int64

Dropping rare classes (< 15 images): ['T']
These classes don't have enough images to train reliably.

Final class distribution:
pest
LM    389
A      64
SM     50
AW     40
Name: count, dtype: int64

PREPARING TRAIN / VALIDATION / TEST DATASET

Checking source images...


Checking files: 100%|██████████| 543/543 [00:00<00:00, 11196.69it/s]

All 543 source images exist.

Removing old dataset:
z:\Projects\Smart-Farming\Crop Identification\notebooks\pest_dataset



Dataset split:
Train: 380 (69.98%)
Val:   81 (14.92%)
Test:  82 (15.10%)

Creating train dataset...


Copying train: 100%|██████████| 380/380 [00:04<00:00, 92.14it/s] 



Creating val dataset...


Copying val: 100%|██████████| 81/81 [00:00<00:00, 82.38it/s] 



Creating test dataset...


Copying test: 100%|██████████| 82/82 [00:00<00:00, 87.90it/s] 



DATASET PREPARATION COMPLETE

Dataset location:
Z:\Projects\Smart-Farming\Crop Identification\notebooks\pest_dataset

Generated dataset:

train/
    A: 45 images
    AW: 28 images
    LM: 272 images
    SM: 35 images

val/
    A: 9 images
    AW: 6 images
    LM: 58 images
    SM: 8 images

test/
    A: 10 images
    AW: 6 images
    LM: 59 images
    SM: 7 images

STARTING YOLOv8 PEST CLASSIFIER TRAINING

Removing previous training artifacts...

Loading YOLOv8 classification model...

Training configuration:
  Dataset       : Z:\Projects\Smart-Farming\Crop Identification\notebooks\pest_dataset
  Epochs        : 30
  Image size    : 224
  Batch size    : 16
  Device        : CUDA GPU 0
  Workers       : 0
  AMP           : False
  Output        : Z:\Projects\Smart-Farming\Crop Identification\notebooks\models\pest_classifier\training\run

Starting training...
New https://pypi.org/project/ultralytics/8.4.123 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.122  Python

## Load and verify saved model

In [9]:
# ============================================================
# VERIFY SAVED MODEL
# ============================================================

print("\n" + "=" * 70)
print("VERIFYING SAVED MODEL")
print("=" * 70)


# ------------------------------------------------------------
# Check final model
# ------------------------------------------------------------

if not FINAL_MODEL_PATH.exists():

    raise FileNotFoundError(
        f"\nFinal model was not found:\n"
        f"{FINAL_MODEL_PATH}"
    )


# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

saved_model = YOLO(
    str(
        FINAL_MODEL_PATH
    )
)


print(
    "\nSaved model loaded successfully!"
)


print(
    f"\nModel path:\n"
    f"{FINAL_MODEL_PATH.resolve()}"
)


print(
    "\nClass names:"
)


print(
    saved_model.names
)


print(
    "\n" + "=" * 70
)


VERIFYING SAVED MODEL

Saved model loaded successfully!

Model path:
Z:\Projects\Smart-Farming\Crop Identification\notebooks\models\pest_classifier\pest_classifier.pt

Class names:
{0: 'A', 1: 'AW', 2: 'LM', 3: 'SM'}



## Show everything saved in models

In [10]:
# ============================================================
# SHOW MODEL DIRECTORY CONTENTS
# ============================================================

print("\n" + "=" * 70)
print("MODEL DIRECTORY CONTENTS")
print("=" * 70)


for path in sorted(
    MODELS_DIR.rglob("*")
):

    if path.is_file():

        size_mb = (
            path.stat().st_size /
            (1024 * 1024)
        )

        print(
            f"{path.relative_to(MODELS_DIR)}"
            f"  ({size_mb:.2f} MB)"
        )


print("=" * 70)


MODEL DIRECTORY CONTENTS
last.pt  (2.83 MB)
pest_classifier.pt  (2.83 MB)
training\run\args.yaml  (0.00 MB)
training\run\confusion_matrix.png  (0.07 MB)
training\run\confusion_matrix_normalized.png  (0.08 MB)
training\run\results.csv  (0.00 MB)
training\run\results.png  (0.11 MB)
training\run\train_batch0.jpg  (0.15 MB)
training\run\train_batch1.jpg  (0.16 MB)
training\run\train_batch2.jpg  (0.16 MB)
training\run\train_batch480.jpg  (0.15 MB)
training\run\train_batch481.jpg  (0.15 MB)
training\run\train_batch482.jpg  (0.15 MB)
training\run\val_batch0_labels.jpg  (0.15 MB)
training\run\val_batch0_pred.jpg  (0.15 MB)
training\run\val_batch1_labels.jpg  (0.16 MB)
training\run\val_batch1_pred.jpg  (0.16 MB)
training\run\val_batch2_labels.jpg  (0.14 MB)
training\run\val_batch2_pred.jpg  (0.14 MB)
training\run\weights\best.pt  (2.83 MB)
training\run\weights\last.pt  (2.83 MB)
